# PrexSyn Baseline

Generates the repeated-PrexSyn-sampling baseline for the benchmarking pipeline.

**Pipeline position:** Stage 2 (Generation) + Stage 4 (Evaluation, baseline arm)

```
ChEMBL NPZ features
      │
      ▼
Stage 1 ─ Build MolSpec for each ChEMBL molecule
      │
      ▼
Stage 2 ─ PrexSyn: generate 256 candidates per spec
          └─ pick K=1 seed (top Tanimoto)
          └─ compute baseline_quality, assign bin
      │
      ▼
Stage 3 ─ Repeated sampling: generate N baseline variants per spec
      │
      ▼
Stage 4 ─ Score baseline variants (scoring_v2)
          └─ synthesizability gate: AiZynthFinder (separate module)
          └─ property conservation: spec-relative desirability + Tanimoto
      │
      ▼
Stage 5 ─ Summarize: hit rate, expansion factor, diversity (bin-stratified)
```

**Outputs:**
- `data/baseline_seeds.json` — seed SMILES + baseline_quality + bin for each spec
- `data/baseline_variants.json` — all N repeated-sampling variants per spec
- `data/baseline_scores.csv` — scored variants (scoring_v2 output)
- `data/baseline_summary.csv` — bin-stratified summary statistics

**Requirements:** `prexsyn` conda environment, PrexSyn API running at `PREXSYN_URL`.

## 0. Configuration

In [ ]:
import pathlib

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT        = pathlib.Path("..")           # repo root (run notebook from notebooks/)
DATA_DIR    = ROOT / "data"

FEATURES_NPZ        = DATA_DIR / "chembl_features.npz"       # pre-computed features (100 mols)
SEEDS_JSON          = DATA_DIR / "baseline_seeds.json"        # Stage 2 output
VARIANTS_JSON       = DATA_DIR / "baseline_variants.json"     # Stage 3 output
SCORES_CSV          = DATA_DIR / "baseline_scores.csv"        # Stage 4 output
SUMMARY_CSV         = DATA_DIR / "baseline_summary.csv"       # Stage 5 output

# ── PrexSyn API ───────────────────────────────────────────────────────────────
PREXSYN_URL = "http://100.65.172.100:8011/sample"  # Tailscale IP of desktop

# ── Generation parameters ────────────────────────────────────────────────────
NUM_SEED_SAMPLES     = 256   # candidates generated for seed selection
NUM_BASELINE_SAMPLES = 256   # repeated-sampling baseline (N; set by pilot study)
LIMIT                = None  # set to small int (e.g. 5) for a quick dry-run

# ── Evaluation parameters ────────────────────────────────────────────────────
TAU_T_LIST  = [0.6, 0.7, 0.85]  # Tanimoto thresholds
TAU_D       = 0.8               # desirability threshold

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Config OK")

## 1. Imports

In [ ]:
import json
import sys
import time
import urllib.error
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Add repo root to path so src/ modules are importable
sys.path.insert(0, str(ROOT))

from src.evaluation.scoring_v2 import (
    BIN_LABELS,
    DEFAULT_TAU_D,
    DEFAULT_TAU_T_LIST,
    make_spec,
    score_batch,
    summarize,
    classify_hits,
    tanimoto_to_spec,
)
from rdkit import Chem

warnings.filterwarnings("ignore")
print("Imports OK")

## 2. Load ChEMBL features

The NPZ file is produced by `featurize_chembl.py` and contains pre-computed
ECFP4, FCFP4, RDKit descriptor, and BRICS fingerprint arrays.
Each row corresponds to one ChEMBL molecule.

In [ ]:
assert FEATURES_NPZ.exists(), (
    f"Features file not found: {FEATURES_NPZ}\n"
    "Run: python src/generation/main.py featurize"
)

data = np.load(FEATURES_NPZ, allow_pickle=True)

smiles_arr       = data["smiles"]
ecfp4_arr        = data["ecfp4"]
fcfp4_arr        = data["fcfp4"]
rdkit_vals_arr   = data["rdkit_desc_values"]
rdkit_names      = data["rdkit_desc_names"].tolist()
brics_fps_arr    = data["brics_fps"]
brics_exists_arr = data["brics_exists"]

N_TOTAL = len(smiles_arr) if LIMIT is None else LIMIT

print(f"Loaded {len(smiles_arr)} molecules from {FEATURES_NPZ}")
print(f"Processing {N_TOTAL} molecules (LIMIT={LIMIT})")
print(f"Feature arrays: ecfp4={ecfp4_arr.shape}, rdkit_vals={rdkit_vals_arr.shape}")

## 3. Build MolSpecs

Each ChEMBL SMILES is converted to a `MolSpec` containing the ECFP4 fingerprint
and target descriptor values (MW, CLogP, TPSA, HBD, HBA, RotBonds).
The ChEMBL molecule itself is not an optimization target.

In [ ]:
specs     = []   # MolSpec objects
spec_smi  = []   # corresponding SMILES strings
skip_idx  = []   # indices where make_spec failed

for i in range(N_TOTAL):
    smi  = str(smiles_arr[i])
    spec = make_spec(smi)
    if spec is None:
        skip_idx.append(i)
        continue
    specs.append(spec)
    spec_smi.append(smi)

print(f"Built {len(specs)} MolSpecs  ({len(skip_idx)} skipped: invalid SMILES)")

# Preview descriptor distribution
desc_df = pd.DataFrame([
    {"mw": s.mw, "clogp": s.clogp, "tpsa": s.tpsa,
     "hbd": s.hbd, "hba": s.hba, "rot_bonds": s.rot_bonds}
    for s in specs
])
desc_df.describe().round(2)

## 4. PrexSyn generation — seed selection (K = 1)

For each spec, call the PrexSyn `/sample` endpoint with `NUM_SEED_SAMPLES=256`
candidates. The top candidate by ECFP4 Tanimoto to the spec is the **seed**
that will be passed to all modification methods.

The seed's Tanimoto (`baseline_quality`) determines the difficulty bin.

In [ ]:
def _call_prexsyn(payload: dict, url: str = PREXSYN_URL, retries: int = 3) -> dict | None:
    """POST to PrexSyn /sample; return parsed JSON or None on failure."""
    for attempt in range(retries):
        try:
            req  = urllib.request.Request(
                url,
                data=json.dumps(payload).encode(),
                headers={"Content-Type": "application/json"},
            )
            resp = urllib.request.urlopen(req, timeout=120)
            return json.loads(resp.read())
        except urllib.error.HTTPError as e:
            print(f"  HTTP {e.code}: {e.read().decode()[:120]}")
            return None
        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"  Failed after {retries} attempts: {exc}")
                return None
    return None


def _build_payload(i: int, num_samples: int) -> dict:
    """Assemble the JSON payload for molecule index i."""
    return {
        "ecfp4":             ecfp4_arr[i].tolist(),
        "fcfp4":             fcfp4_arr[i].tolist(),
        "rdkit_desc_values": rdkit_vals_arr[i].tolist(),
        "rdkit_desc_names":  rdkit_names,
        "brics_fps":         brics_fps_arr[i].tolist(),
        "brics_exists":      brics_exists_arr[i].tolist(),
        "source_smiles":     str(smiles_arr[i]),
        "num_samples":       num_samples,
    }


print("Payload builder ready.")

In [ ]:
seeds = []   # list of dicts: {spec_smiles, seed_smiles, baseline_quality, quality_bin, raw_candidates}

# Map from original array index → spec index (skipping invalid SMILES)
valid_indices = [i for i in range(N_TOTAL) if i not in skip_idx]

for spec_idx, arr_idx in enumerate(tqdm(valid_indices, desc="PrexSyn seed generation")):
    spec = specs[spec_idx]

    payload  = _build_payload(arr_idx, num_samples=NUM_SEED_SAMPLES)
    response = _call_prexsyn(payload)

    if response is None or not response.get("generated_smiles"):
        seeds.append({
            "spec_smiles":      spec_smi[spec_idx],
            "seed_smiles":      None,
            "baseline_quality": 0.0,
            "quality_bin":      "<0.5",
            "num_candidates":   0,
            "raw_candidates":   [],
        })
        continue

    candidates = response["generated_smiles"]

    # Pick K=1: highest Tanimoto to spec fingerprint
    best_smi, best_tan = None, -1.0
    for smi in candidates:
        if "." in smi:
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        tan = tanimoto_to_spec(mol, spec)
        if tan > best_tan:
            best_tan, best_smi = tan, smi

    # Assign difficulty bin
    from src.evaluation.scoring_v2 import _assign_bin
    qbin = _assign_bin(best_tan) if best_smi else "<0.5"

    seeds.append({
        "spec_smiles":      spec_smi[spec_idx],
        "seed_smiles":      best_smi,
        "baseline_quality": round(best_tan, 4),
        "quality_bin":      qbin,
        "num_candidates":   len(candidates),
        "raw_candidates":   candidates,   # kept for diversity analysis
    })

SEEDS_JSON.write_text(json.dumps(seeds, indent=2))
print(f"Saved {len(seeds)} seed records → {SEEDS_JSON}")

In [ ]:
# ── Baseline quality distribution ────────────────────────────────────────────
seeds_df = pd.DataFrame(seeds)
seeds_df = seeds_df[seeds_df["seed_smiles"].notna()]

print(f"Valid seeds: {len(seeds_df)} / {len(seeds)}")
print("\nBaseline quality distribution:")
print(seeds_df["baseline_quality"].describe().round(3))
print("\nBin counts:")
print(seeds_df["quality_bin"].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(7, 3))
seeds_df["baseline_quality"].hist(bins=20, ax=ax, color="steelblue", edgecolor="white")
for threshold in [0.5, 0.7, 0.85]:
    ax.axvline(threshold, color="tomato", linewidth=1.2, linestyle="--")
ax.set_xlabel("Baseline quality (Tanimoto, seed vs spec)")
ax.set_ylabel("Count")
ax.set_title("PrexSyn baseline quality distribution")
plt.tight_layout()
plt.show()

## 5. Repeated-sampling baseline generation (N draws)

For each spec, generate `NUM_BASELINE_SAMPLES` additional PrexSyn candidates
from the same property specification. These form the repeated-sampling baseline
that all modification methods are compared against.

The same spec payload is reused; PrexSyn's sampling is stochastic so the
returned set differs from the seed-generation batch.

In [ ]:
# Build index: spec_idx → arr_idx (needed for payload construction)
arr_idx_map = {spec_idx: arr_idx for spec_idx, arr_idx in enumerate(valid_indices)}

baseline_variants = []  # list of dicts: {spec_smiles, baseline_quality, quality_bin, variants}

for spec_idx, row in enumerate(tqdm(seeds, desc="Repeated-sampling baseline")):
    if row["seed_smiles"] is None:
        continue

    arr_idx  = arr_idx_map[spec_idx]
    payload  = _build_payload(arr_idx, num_samples=NUM_BASELINE_SAMPLES)
    response = _call_prexsyn(payload)

    variants = response.get("generated_smiles", []) if response else []

    # Deduplicate by canonical SMILES; exclude seed
    seen     = {row["seed_smiles"]}
    deduped  = []
    for smi in variants:
        canon = Chem.MolToSmiles(Chem.MolFromSmiles(smi)) if Chem.MolFromSmiles(smi) else None
        if canon and canon not in seen:
            seen.add(canon)
            deduped.append(canon)

    baseline_variants.append({
        "spec_smiles":      row["spec_smiles"],
        "baseline_quality": row["baseline_quality"],
        "quality_bin":      row["quality_bin"],
        "seed_smiles":      row["seed_smiles"],
        "variants":         deduped,
        "n_raw":            len(variants),
        "n_deduped":        len(deduped),
    })

VARIANTS_JSON.write_text(json.dumps(baseline_variants, indent=2))
total_variants = sum(e["n_deduped"] for e in baseline_variants)
print(f"Saved {len(baseline_variants)} specs, {total_variants} total variants → {VARIANTS_JSON}")

In [ ]:
# ── Variant yield summary ──────────────────────────────────────────────────────
yield_df = pd.DataFrame([
    {"quality_bin": e["quality_bin"],
     "n_raw":       e["n_raw"],
     "n_deduped":   e["n_deduped"]}
    for e in baseline_variants
])

print("Variant yield by difficulty bin:")
print(
    yield_df.groupby("quality_bin")[["n_raw", "n_deduped"]]
    .agg(["mean", "min", "max"])
    .round(1)
)

## 6. Synthesizability gate (AiZynthFinder)

Each baseline variant must pass retrosynthetic analysis before property
conservation scoring. The `synthesizability.py` module wraps AiZynthFinder.

> **Note:** AiZynthFinder runs in the `prexsyn` environment. This cell
> calls `src/evaluation/synthesizability.py` directly if the model files
> are present; otherwise it skips the gate and records all variants as
> unverified (for dry-run / development purposes).

In [ ]:
AIZYNTHFINDER_CONFIG = DATA_DIR / "aizynthfinder" / "config.yml"

USE_AIZYNTHFINDER = AIZYNTHFINDER_CONFIG.exists()
if USE_AIZYNTHFINDER:
    from src.evaluation.synthesizability import score_batch as synth_score_batch
    print(f"AiZynthFinder config found: {AIZYNTHFINDER_CONFIG}")
else:
    print(
        f"[WARN] AiZynthFinder config not found at {AIZYNTHFINDER_CONFIG}.\n"
        "       Skipping synthesizability gate — all variants treated as synthesizable.\n"
        "       Set AIZYNTHFINDER_CONFIG to enable Gate 1."
    )


def filter_synthesizable(variants: list[str]) -> list[str]:
    """
    Returns the subset of variants that AiZynthFinder deems synthesizable.
    Falls back to returning all variants if AiZynthFinder is not configured.
    """
    if not USE_AIZYNTHFINDER:
        return variants   # gate skipped in dev mode

    results = synth_score_batch(variants, str(AIZYNTHFINDER_CONFIG))
    return [r["smiles"] for r in results if r["is_solved"]]

In [ ]:
# Apply synthesizability gate to all baseline variants
synth_filtered = []   # same structure as baseline_variants, with variants filtered

for entry in tqdm(baseline_variants, desc="Synthesizability gate"):
    filtered = filter_synthesizable(entry["variants"])
    synth_filtered.append({**entry, "variants": filtered, "n_synth": len(filtered)})

total_synth = sum(e["n_synth"] for e in synth_filtered)
total_raw   = sum(e["n_deduped"] for e in baseline_variants)
print(f"After synthesizability gate: {total_synth} / {total_raw} variants pass "
      f"({'N/A (gate skipped)' if not USE_AIZYNTHFINDER else f'{100*total_synth/total_raw:.1f}%'})")

## 7. Score with scoring_v2

Each synthesizable baseline variant is scored against its property specification:
- **Substructural conservation:** ECFP4 Tanimoto vs. spec fingerprint  
- **Physicochemical conservation:** spec-relative desirability (geometric mean of 6 descriptor deltas)

In [ ]:
# Build a SMILES → MolSpec lookup for fast access
spec_lookup = {smi: spec for smi, spec in zip(spec_smi, specs)}

all_score_dfs = []

for entry in tqdm(synth_filtered, desc="Scoring baseline variants"):
    spec = spec_lookup.get(entry["spec_smiles"])
    if spec is None or not entry["variants"]:
        continue

    batch_df = score_batch(
        variants         = entry["variants"],
        spec             = spec,
        baseline_quality = entry["baseline_quality"],
        method           = "baseline",
    )
    if not batch_df.empty:
        all_score_dfs.append(batch_df)

scores_df = pd.concat(all_score_dfs, ignore_index=True) if all_score_dfs else pd.DataFrame()

scores_df.to_csv(SCORES_CSV, index=False)
print(f"Scored {len(scores_df)} variants → {SCORES_CSV}")
scores_df.head()

## 8. Summary statistics

Bin-stratified hit rates at all three Tanimoto thresholds.  
These values define the baseline against which all modification methods are measured.

In [ ]:
if scores_df.empty:
    print("No scored variants — check upstream steps.")
else:
    summary_df = summarize(scores_df, tau_t_list=TAU_T_LIST, tau_d=TAU_D)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    print(f"Summary ({len(summary_df)} rows) → {SUMMARY_CSV}")
    display(summary_df)

In [ ]:
# ── Hit rate by bin × threshold ───────────────────────────────────────────────
if not scores_df.empty:
    fig, axes = plt.subplots(1, len(TAU_T_LIST), figsize=(5 * len(TAU_T_LIST), 4), sharey=True)

    for ax, tau_t in zip(axes, TAU_T_LIST):
        sub = summary_df[summary_df["tau_t"] == tau_t]
        ax.bar(sub["quality_bin"], sub["hit_rate"], color="steelblue", edgecolor="white")
        ax.set_title(f"τ_t = {tau_t}")
        ax.set_xlabel("Baseline quality bin")
        ax.set_ylabel("Hit rate")
        ax.set_ylim(0, 1)
        for bar, val in zip(ax.patches, sub["hit_rate"]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9
            )

    fig.suptitle("Baseline hit rate by difficulty bin", fontsize=13)
    plt.tight_layout()
    plt.savefig(DATA_DIR / "baseline_hit_rates.png", dpi=150)
    plt.show()

In [ ]:
# ── Score distributions ───────────────────────────────────────────────────────
if not scores_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    for qbin, grp in scores_df.groupby("quality_bin"):
        grp["tanimoto"].plot.kde(ax=axes[0], label=qbin)
        grp["desirability"].plot.kde(ax=axes[1], label=qbin)

    axes[0].set_title("Tanimoto distribution by bin")
    axes[0].set_xlabel("Tanimoto (variant vs spec)")
    for tau_t in TAU_T_LIST:
        axes[0].axvline(tau_t, linewidth=0.8, linestyle="--", color="gray")

    axes[1].set_title("Desirability distribution by bin")
    axes[1].set_xlabel("Spec-relative desirability")
    axes[1].axvline(TAU_D, linewidth=0.8, linestyle="--", color="gray", label=f"τ_d={TAU_D}")

    for ax in axes:
        ax.legend(title="Bin", fontsize=8)
        ax.set_ylabel("Density")

    plt.tight_layout()
    plt.savefig(DATA_DIR / "baseline_score_distributions.png", dpi=150)
    plt.show()

## 9. Saturation analysis

Check whether additional PrexSyn samples continue to find new unique hits, or
whether the baseline saturates. This informs the choice of N for all methods.

For each spec, compute cumulative unique hits as a function of the number of
variants considered (in order of generation).

In [ ]:
if not scores_df.empty:
    TAU_T_PILOT = 0.6  # use most permissive threshold for saturation check

    # For each spec, track cumulative unique hits as N increases
    saturation_curves = []

    for spec_smiles, grp in scores_df.groupby("spec_smiles"):
        grp = grp.reset_index(drop=True)
        grp["is_hit"] = classify_hits(grp, tau_t=TAU_T_PILOT, tau_d=TAU_D)

        seen_hits  = set()
        curve      = []
        for _, row in grp.iterrows():
            if row["is_hit"]:
                seen_hits.add(row["variant_smiles"])
            curve.append(len(seen_hits))

        saturation_curves.append(curve)

    # Pad curves to the same length and average
    max_len    = max(len(c) for c in saturation_curves)
    padded     = [c + [c[-1]] * (max_len - len(c)) for c in saturation_curves]
    mean_curve = np.mean(padded, axis=0)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(range(1, max_len + 1), mean_curve, color="steelblue")
    ax.set_xlabel("Number of variants sampled")
    ax.set_ylabel("Mean cumulative unique hits")
    ax.set_title(f"Baseline saturation curve (τ_t={TAU_T_PILOT}, τ_d={TAU_D})")
    plt.tight_layout()
    plt.savefig(DATA_DIR / "baseline_saturation.png", dpi=150)
    plt.show()

    # Marginal gain in last 10% of samples
    cutoff      = int(0.9 * max_len)
    marginal_10 = mean_curve[-1] - mean_curve[cutoff]
    print(f"Mean unique hits at N={max_len}: {mean_curve[-1]:.2f}")
    print(f"Marginal gain in last 10% of samples: {marginal_10:.2f} hits")

## 10. Export seed file for modification methods

Export the seed SMILES for each spec in the format expected by the
modification method notebooks (CReM, mmpdb, JT-VAE, LibINVENT).

In [ ]:
# Format for modification method notebooks:
# [
#   {
#     "spec_smiles":      "...",
#     "seed_smiles":      "...",
#     "baseline_quality": 0.72,
#     "quality_bin":      "0.7-0.85",
#     "methods": {"baseline": ["...", ...]}
#   },
#   ...
# ]

seeds_for_methods = []

for entry in synth_filtered:
    seed_row = next(
        (s for s in seeds if s["spec_smiles"] == entry["spec_smiles"]), None
    )
    if seed_row is None or seed_row["seed_smiles"] is None:
        continue

    seeds_for_methods.append({
        "spec_smiles":      entry["spec_smiles"],
        "seed_smiles":      seed_row["seed_smiles"],
        "baseline_quality": entry["baseline_quality"],
        "quality_bin":      entry["quality_bin"],
        "methods": {
            "baseline": entry["variants"]   # synthesizability-filtered
        },
    })

SEEDS_FOR_METHODS_JSON = DATA_DIR / "seeds_for_methods.json"
SEEDS_FOR_METHODS_JSON.write_text(json.dumps(seeds_for_methods, indent=2))

print(f"Exported {len(seeds_for_methods)} specs with seed + baseline variants")
print(f"→ {SEEDS_FOR_METHODS_JSON}")
print("\nThis file is the input for all modification method notebooks.")

## 11. Final summary

Print a concise overview of what was produced.

In [ ]:
print("=" * 60)
print("PrexSyn Baseline — Final Summary")
print("=" * 60)
print(f"  Specs processed:          {len(seeds_for_methods)}")
print(f"  Total baseline variants:  {total_raw}")
print(f"  After synth gate:         {total_synth}")
print(f"  Scored variants:          {len(scores_df)}")
print()
print("Outputs:")
for path in [SEEDS_JSON, VARIANTS_JSON, SCORES_CSV, SUMMARY_CSV, SEEDS_FOR_METHODS_JSON]:
    size = path.stat().st_size if path.exists() else 0
    print(f"  {path.name:<35s}  {size:>8,} bytes")
print()

if not scores_df.empty:
    print("Hit rates at τ_d = 0.8:")
    pivot = summary_df.pivot_table(
        index="quality_bin", columns="tau_t", values="hit_rate"
    ).round(3)
    print(pivot.to_string())